<a href="https://colab.research.google.com/github/Sud2025/ai-content-evaluator/blob/main/AI_Content_Evaluator_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install langgraph

In [7]:
from langgraph.graph import StateGraph
from typing import TypedDict, List, Dict, Any

# Define a TypedDict for the graph state to ensure proper state management
class GraphState(TypedDict):
    query: str
    data: List[Dict[str, Any]]
    filtered: List[Dict[str, Any]]
    result: List[Dict[str, Any]]

def input_node(state: GraphState):
    return {"query": state.get("query", "")}

def retrieve_node(state: GraphState):
    data = [
        {"title": "Movie Trailer 1", "channel": "Official Studio", "views": 1000000},
        {"title": "Movie Trailer 2", "channel": "Random Channel", "views": 50000},
        {"title": "Movie Trailer 3", "channel": "Official Studio", "views": 750000},
    ]
    # This will merge 'data' into the existing state
    return {"data": data}

def filter_node(state: GraphState):
    # Now 'data' should be guaranteed to be in the state
    return {"filtered": [d for d in state["data"] if d["channel"] == "Official Studio"]}

def rank_node(state: GraphState):
    return {"result": sorted(state["filtered"], key=lambda x: x["views"], reverse=True)}

builder = StateGraph(GraphState)

builder.add_node("input", input_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("filter", filter_node)
builder.add_node("rank", rank_node)

builder.set_entry_point("input")

builder.add_edge("input", "retrieve")
builder.add_edge("retrieve", "filter")
builder.add_edge("filter", "rank")

graph = builder.compile()

result = graph.invoke({"query": "top trailers"})
print(result["result"])


[{'title': 'Movie Trailer 1', 'channel': 'Official Studio', 'views': 1000000}, {'title': 'Movie Trailer 3', 'channel': 'Official Studio', 'views': 750000}]
